# Preparation of GWR - Data

This Jupyter notebook does the following:

- Download the latest building-register (GWR) GeoJSON data for the Canton of Zurich.
- Flatten building features into a pandas DataFrame and separate their Swiss LV95 coordinates.
- Convert building coordinates from Swiss LV95 to WGS84 latitude and longitude.
- Add readable building-category and building-class descriptions from Excel lookup tables.
- Display a sample of buildings in Greifensee on an interactive map.
- Create a full Wädenswil building subset and plot the buildings on an interactive map.

## Libraries and Settings

In [12]:
# Libraries
import os
import json
from pathlib import Path
import folium
import requests
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from datetime import date

# Keep the GWR inputs and outputs in this lesson folder.
lesson_dir = Path.cwd()
if not (lesson_dir / 'gkat_translator.xlsx').is_file():
    if (lesson_dir / 'Week_11' / '05_Python_GWR_Data').is_dir():
        lesson_dir = lesson_dir / 'Week_11' / '05_Python_GWR_Data'
    elif (lesson_dir / '05_Python_GWR_Data').is_dir():
        lesson_dir = lesson_dir / '05_Python_GWR_Data'
    else:
        raise FileNotFoundError('Could not locate the 05_Python_GWR_Data lesson folder.')
os.chdir(lesson_dir)

# Ignore warnings
import warnings
warnings.filterwarnings("ignore")

print(os.getcwd())

/workspaces/ARM-Stejsnat/Week_11/05_Python_GWR_Data


## Download GWR data

Source: https://public.madd.bfs.admin.ch

In [13]:
# Download the latest file only when it is not already available locally.
url = "https://public.madd.bfs.admin.ch/buildings_zh.geojson"
geojson_path = Path('buildings_zh.geojson')
if not geojson_path.is_file():
    with requests.get(url, stream=True, timeout=(15, 300)) as response:
        response.raise_for_status()
        with geojson_path.open('wb') as file:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    file.write(chunk)
print(f'Using {geojson_path} ({geojson_path.stat().st_size:,} bytes)')

Using buildings_zh.geojson (317,214,805 bytes)


## Import latest GWR Data

In [4]:
# Load json file
with open('buildings_zh.geojson') as f:
    data = json.load(f)

# Flatten nested json data
df_orig = pd.json_normalize(data, record_path=['features'])
df_orig

# Remove prefix
df_orig.columns = df_orig.columns.str.replace('properties.', '')
df_orig.columns = df_orig.columns.str.replace('geometry.', '')

# Create copy
df = df_orig.copy()
df.head()

,type,type,coordinates,egid,buildingStatus,buildingCategory,buildingClass,municipalityNumber,municipalityName,canton,...,heating2_informationSource,heating2_revisionDate,hotwater1_heatGenerator,hotwater1_energySource,hotwater1_informationSource,hotwater1_revisionDate,hotwater2_heatGenerator,hotwater2_energySource,hotwater2_informationSource,hotwater2_revisionDate
0,Feature,Point,"[2676490.0, 1235841.0]",1,1004,1040.0,1271.0,2,Affoltern am Albis,ZH,...,860.0,2001-11-29,7630.0,7530.0,860.0,2001-11-29,7600.0,7500.0,860.0,2001-11-29
1,Feature,Point,"[2676541.16, 1235979.043]",2,1007,1020.0,1110.0,2,Affoltern am Albis,ZH,...,NaN,NaN,7630.0,7530.0,860.0,2001-11-29,NaN,NaN,NaN,NaN
2,Feature,Point,"[2676529.561, 1236015.749]",3,1004,1020.0,1110.0,2,Affoltern am Albis,ZH,...,869.0,2026-02-20,7610.0,7501.0,869.0,2026-02-20,7600.0,7500.0,869.0,2026-02-20
3,Feature,Point,"[2676512.712, 1235974.383]",4,1004,1020.0,1110.0,2,Affoltern am Albis,ZH,...,860.0,2001-11-29,7610.0,7501.0,869.0,2026-09-11,7600.0,7500.0,860.0,2001-11-29
4,Feature,Point,"[2676502.104, 1236014.389]",5,1004,1020.0,1110.0,2,Affoltern am Albis,ZH,...,860.0,2001-11-29,7610.0,7501.0,865.0,2026-07-09,7600.0,7500.0,860.0,2001-11-29


## Separate Swiss LV95 coordinates

In [5]:
# Separate coordinates
df['x_coords'] = pd.DataFrame(df['coordinates'].tolist(), columns=['x_coord', 'y_coord'])['x_coord']
df['y_coords'] = pd.DataFrame(df['coordinates'].tolist(), columns=['x_coord', 'y_coord'])['y_coord']

# Remove column 'coordinates'
df = df.drop(['coordinates'], axis=1)
df

,type,type,egid,buildingStatus,buildingCategory,buildingClass,municipalityNumber,municipalityName,canton,energyReferenceArea,...,hotwater1_heatGenerator,hotwater1_energySource,hotwater1_informationSource,hotwater1_revisionDate,hotwater2_heatGenerator,hotwater2_energySource,hotwater2_informationSource,hotwater2_revisionDate,x_coords,y_coords
0,Feature,Point,1,1004,1040.0,1271.0,2,Affoltern am Albis,ZH,NaN,...,7630.0,7530.0,860.0,2001-11-29,7600.0,7500.0,860.0,2001-11-29,2676490.000,1235841.000
1,Feature,Point,2,1007,1020.0,1110.0,2,Affoltern am Albis,ZH,NaN,...,7630.0,7530.0,860.0,2001-11-29,NaN,NaN,NaN,NaN,2676541.160,1235979.043
2,Feature,Point,3,1004,1020.0,1110.0,2,Affoltern am Albis,ZH,200.0,...,7610.0,7501.0,869.0,2026-02-20,7600.0,7500.0,869.0,2026-02-20,2676529.561,1236015.749
3,Feature,Point,4,1004,1020.0,1110.0,2,Affoltern am Albis,ZH,200.0,...,7610.0,7501.0,869.0,2026-09-11,7600.0,7500.0,860.0,2001-11-29,2676512.712,1235974.383
4,Feature,Point,5,1004,1020.0,1110.0,2,Affoltern am Albis,ZH,NaN,...,7610.0,7501.0,865.0,2026-07-09,7600.0,7500.0,860.0,2001-11-29,2676502.104,1236014.389
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
400751,Feature,Point,504233631,1004,1060.0,1274.0,136,Langnau am Albis,ZH,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2683209.226,1237783.390
400752,Feature,Point,504233632,1004,1060.0,1274.0,136,Langnau am Albis,ZH,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2682598.328,1237176.167
400753,Feature,Point,504233633,1004,1060.0,1274.0,136,Langnau am Albis,ZH,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2682281.367,1236777.247
400754,Feature,Point,504233634,1004,1060.0,1274.0,136,Langnau am Albis,ZH,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2682305.267,1238164.811


## Create WGS84 coordinates

In [6]:
# Create geodataframe and calculate latitude and longitude
gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df['x_coords'], df['y_coords']), crs="EPSG:2056")

# Convert the Swiss LV95 coordinates to lat & lon
gdf = gdf.to_crs(epsg=4326)

# Get Latitude and Longitude
df['latitude'] = gdf['geometry'].y
df['longitude'] = gdf['geometry'].x

# Show data
df


,type,type,egid,buildingStatus,buildingCategory,buildingClass,municipalityNumber,municipalityName,canton,energyReferenceArea,...,hotwater1_informationSource,hotwater1_revisionDate,hotwater2_heatGenerator,hotwater2_energySource,hotwater2_informationSource,hotwater2_revisionDate,x_coords,y_coords,latitude,longitude
0,Feature,Point,1,1004,1040.0,1271.0,2,Affoltern am Albis,ZH,NaN,...,860.0,2001-11-29,7600.0,7500.0,860.0,2001-11-29,2676490.000,1235841.000,47.269041,8.449423
1,Feature,Point,2,1007,1020.0,1110.0,2,Affoltern am Albis,ZH,NaN,...,860.0,2001-11-29,NaN,NaN,NaN,NaN,2676541.160,1235979.043,47.270277,8.450122
2,Feature,Point,3,1004,1020.0,1110.0,2,Affoltern am Albis,ZH,200.0,...,869.0,2026-02-20,7600.0,7500.0,869.0,2026-02-20,2676529.561,1236015.749,47.270608,8.449975
3,Feature,Point,4,1004,1020.0,1110.0,2,Affoltern am Albis,ZH,200.0,...,869.0,2026-09-11,7600.0,7500.0,860.0,2001-11-29,2676512.712,1235974.383,47.270238,8.449746
4,Feature,Point,5,1004,1020.0,1110.0,2,Affoltern am Albis,ZH,NaN,...,865.0,2026-07-09,7600.0,7500.0,860.0,2001-11-29,2676502.104,1236014.389,47.270599,8.449612
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
400751,Feature,Point,504233631,1004,1060.0,1274.0,136,Langnau am Albis,ZH,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,2683209.226,1237783.390,47.285698,8.538564
400752,Feature,Point,504233632,1004,1060.0,1274.0,136,Langnau am Albis,ZH,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,2682598.328,1237176.167,47.280313,8.530378
400753,Feature,Point,504233633,1004,1060.0,1274.0,136,Langnau am Albis,ZH,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,2682281.367,1236777.247,47.276765,8.526116
400754,Feature,Point,504233634,1004,1060.0,1274.0,136,Langnau am Albis,ZH,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,2682305.267,1238164.811,47.289242,8.526686


## Match descriptions to buildingCategory &	buildingClass	

In [7]:
# Read .xslx file with the data
df_gwr_cat = pd.read_excel('gkat_translator.xlsx', sheet_name='buildingCategory')
df_gwr_class = pd.read_excel('gkat_translator.xlsx', sheet_name='buildingClass')

# Merge the DataFrames on 'buildingCategory'
df_merged = pd.merge(df, df_gwr_cat, on='buildingCategory', how='left')
df_merged = pd.merge(df_merged, df_gwr_class, on='buildingClass', how='left')

# Show data
df_merged[['egid', 'buildingCategory', 'buildingClass', 'buildingCategoryDesc', 'buildingClassDesc']].head()

,egid,buildingCategory,buildingClass,buildingCategoryDesc,buildingClassDesc
0,1,1040.0,1271.0,Gebäude mit teilweiser Wohnnutzung,Landwirtschaftliche Betriebsgebäude
1,2,1020.0,1110.0,Gebäude mit ausschliesslicher Wohnnutzung,Gebäude mit einer Wohnung
2,3,1020.0,1110.0,Gebäude mit ausschliesslicher Wohnnutzung,Gebäude mit einer Wohnung
3,4,1020.0,1110.0,Gebäude mit ausschliesslicher Wohnnutzung,Gebäude mit einer Wohnung
4,5,1020.0,1110.0,Gebäude mit ausschliesslicher Wohnnutzung,Gebäude mit einer Wohnung


## Plot subset of buildings

In [8]:
# Subset
df_sub = df_merged.loc[df['municipalityName'] == 'Greifensee'].sample(100).dropna()

# Create the map
m = folium.Map(location=[df_sub['latitude'].mean(), df_sub['longitude'].mean()], zoom_start=15, tiles='OpenStreetMap')

# Add points to the map
for idx, row in df_sub.iterrows():
    folium.Marker(location=([row['latitude'], 
                            row['longitude']]),
                  popup=row['buildingClassDesc']).add_to(m)

# Display the map
m

## Buildings in Wädenswil

Create a subset of all buildings in Wädenswil and plot every building with valid coordinates.

In [14]:
# Keep all Wädenswil buildings in the subset and CSV, including rows without coordinates.
df_waedenswil = df_merged.loc[df_merged['municipalityName'].eq('Wädenswil')].copy()
if df_waedenswil.empty:
    raise ValueError('No GWR buildings were found for Wädenswil.')

df_waedenswil.to_csv('buildings_waedenswil.csv', index=False)

# Only rows with coordinates can be shown on a map.
map_data = df_waedenswil.dropna(subset=['latitude', 'longitude'])
if map_data.empty:
    raise ValueError('Wädenswil buildings were found, but none have valid coordinates.')

gdf_waedenswil = gpd.GeoDataFrame(
    map_data[['egid', 'buildingCategoryDesc', 'buildingClassDesc']].copy(),
    geometry=gpd.points_from_xy(map_data['longitude'], map_data['latitude']),
    crs='EPSG:4326'
)

m_waedenswil = folium.Map(
    location=[map_data['latitude'].mean(), map_data['longitude'].mean()],
    zoom_start=13,
    tiles='OpenStreetMap'
)
folium.GeoJson(
    gdf_waedenswil.to_json(),
    name='Wädenswil buildings',
    marker=folium.CircleMarker(
        radius=2,
        color='#1565c0',
        fill=True,
        fill_opacity=0.7
    ),
    tooltip=folium.GeoJsonTooltip(
        fields=['egid', 'buildingCategoryDesc', 'buildingClassDesc'],
        aliases=['EGID', 'Building category', 'Building class']
    )
).add_to(m_waedenswil)
folium.LayerControl().add_to(m_waedenswil)
m_waedenswil.fit_bounds([
    [map_data['latitude'].min(), map_data['longitude'].min()],
    [map_data['latitude'].max(), map_data['longitude'].max()]
])
m_waedenswil.save('buildings_map_waedenswil.html')

# Save a static map screenshot with the Wädenswil municipality boundary.
municipalities = gpd.read_file('../02_Python_Geocoding_Addresses/GEN_A4_GEMEINDEN_2019_epsg4326.json').to_crs('EPSG:4326')
waedenswil_boundary = municipalities.loc[municipalities['BFS'].eq(293)]
if waedenswil_boundary.empty:
    raise ValueError('Wädenswil was not found in the municipality boundaries.')
fig, ax = plt.subplots(figsize=(10, 8))
waedenswil_boundary.plot(ax=ax, color='#f3f5f7', edgecolor='#455a64', linewidth=1)
gdf_waedenswil.plot(ax=ax, color='#1565c0', markersize=1, alpha=0.45)
ax.set_title(f'Buildings in Wädenswil (n={len(map_data):,})')
ax.set_axis_off()
fig.tight_layout()
fig.savefig(Path('..') / 'buildings_map_waedenswil.png', dpi=200, bbox_inches='tight')
plt.close(fig)


print(f'Buildings in Wädenswil: {len(df_waedenswil):,}')
print(f'Buildings plotted: {len(map_data):,}')
print(f'Buildings missing coordinates: {len(df_waedenswil) - len(map_data):,}')
m_waedenswil

Buildings in Wädenswil: 7,335
Buildings plotted: 7,335
Buildings missing coordinates: 0


## Existing one-apartment buildings in Zurich

Select buildings with GKLAS 1110 and existing building status (1004), count them, and map the selection.

In [15]:
# GWR status 1004 means existing; buildingClass 1110 means one apartment.
# BFS municipality number 261 identifies Zurich city.
df_zuerich_1110_existing = df_merged.loc[
    df_merged['municipalityNumber'].eq(261)
    & df_merged['buildingStatus'].eq(1004)
    & df_merged['buildingClass'].eq(1110)
].copy()
if df_zuerich_1110_existing.empty:
    raise ValueError('No existing GKLAS 1110 buildings were found in Zurich.')

df_zuerich_1110_existing.to_csv('buildings_zuerich_gklas1110_existing.csv', index=False)
print(f'Existing buildings with one apartment in Zurich (GKLAS 1110): {len(df_zuerich_1110_existing):,}')

# Plot all selected buildings with valid coordinates and save the requested screenshot.
zuerich_map_data = df_zuerich_1110_existing.dropna(subset=['latitude', 'longitude'])
municipality_zuerich = municipalities.loc[municipalities['BFS'].eq(261)]
if municipality_zuerich.empty:
    raise ValueError('Zurich was not found in the municipality boundaries.')
if zuerich_map_data.empty:
    raise ValueError('Selected Zurich buildings have no valid coordinates.')

gdf_zuerich_selection = gpd.GeoDataFrame(
    zuerich_map_data[['egid']].copy(),
    geometry=gpd.points_from_xy(zuerich_map_data['longitude'], zuerich_map_data['latitude']),
    crs='EPSG:4326'
)
fig, ax = plt.subplots(figsize=(10, 8))
municipality_zuerich.plot(ax=ax, color='#f3f5f7', edgecolor='#455a64', linewidth=1)
gdf_zuerich_selection.plot(ax=ax, color='#d1495b', markersize=0.8, alpha=0.4)
ax.set_title(f'Existing one-apartment buildings in Zurich (n={len(df_zuerich_1110_existing):,})')
ax.set_axis_off()
fig.tight_layout()
fig.savefig(Path('..') / 'bldg_selection_zuerich.png', dpi=200, bbox_inches='tight')
plt.close(fig)
print(f'Buildings mapped: {len(zuerich_map_data):,}')
df_zuerich_1110_existing[['egid', 'buildingStatus', 'buildingClass', 'buildingClassDesc']].head()

Existing buildings with one apartment in Zurich (GKLAS 1110): 9,253


Buildings mapped: 9,253


,egid,buildingStatus,buildingClass,buildingClassDesc
118652,140010,1004,1110.0,Gebäude mit einer Wohnung
118767,140151,1004,1110.0,Gebäude mit einer Wohnung
118819,140207,1004,1110.0,Gebäude mit einer Wohnung
118821,140209,1004,1110.0,Gebäude mit einer Wohnung
118825,140213,1004,1110.0,Gebäude mit einer Wohnung


### Jupyter notebook --footer info-- (please always provide this at the end of each notebook)

In [9]:
import os
import platform
import socket
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')

-----------------------------------
POSIX
Linux | 6.8.0-1070-azure
Datetime: 2026-11-27 08:29:16
Python Version: 3.11.18
-----------------------------------
